# Serverless coverage-validity validation — 3 coverage functions

Proves the light-tier coverage-validity functions work end-to-end on real Serverless (e2-demo / oauth-fe).

**Three stages:**
1. `gbx_st_coverageisvalid` — valid adjacent-polygon set returns True; overlapping set returns False
2. `gbx_st_coverageinvalidedges` — valid coverage has empty invalid-edges; invalid coverage has non-empty edges
3. `vx.coverage_simplify` — N→N rows, all columns preserved, shared collinear vertex removed

In [ ]:
import datetime
import json

from shapely import box, from_wkb, to_wkb
from shapely.geometry import Polygon

results = {
    "run_date": datetime.datetime.utcnow().isoformat() + "Z",
    "functions_tested": 3,
    "stages": {},
}

print("Coverage-validity validation: gbx_st_coverageisvalid, gbx_st_coverageinvalidedges, vx.coverage_simplify")

In [ ]:
from databricks.labs.gbx.pyvx import functions as vx

vx.register(spark, only=["gbx_st_coverageisvalid", "gbx_st_coverageinvalidedges"])
print("Registered: gbx_st_coverageisvalid, gbx_st_coverageinvalidedges")

In [ ]:
# ---------------------------------------------------------------------------
# Build coverage test data.
# Valid coverage: 4 unit squares tiling a 2x2 grid (shared edges, no gaps or overlaps).
# Invalid coverage: 2 overlapping rectangles (overlap region is the invalid part).
# ---------------------------------------------------------------------------
polys_valid = [
    box(0, 0, 1, 1),
    box(1, 0, 2, 1),
    box(0, 1, 1, 2),
    box(1, 1, 2, 2),
]

polys_invalid = [
    box(0, 0, 2, 2),
    box(1, 0, 3, 2),
]

rows = (
    [("valid", to_wkb(p)) for p in polys_valid]
    + [("invalid", to_wkb(p)) for p in polys_invalid]
)
df_cov = spark.createDataFrame(rows, "cov_id string, geom binary")
df_cov.createOrReplaceTempView("cov")
print(f"Coverage data: {len(polys_valid)} polys (valid cov) + {len(polys_invalid)} polys (invalid cov)")

In [ ]:
# ---------------------------------------------------------------------------
# Stages 1 + 2: coverageisvalid and coverageinvalidedges via SQL GROUP BY.
# Both grouped-aggs run in the same query; results split by cov_id.
# ---------------------------------------------------------------------------
cov_result = spark.sql("""
    SELECT cov_id,
           gbx_st_coverageisvalid(geom, 0.0)       AS ok,
           gbx_st_coverageinvalidedges(geom, 0.0)   AS bad_edges
    FROM cov
    GROUP BY cov_id
    ORDER BY cov_id
""").collect()

by_id = {row["cov_id"]: row for row in cov_result}

# --- Stage 1: coverageisvalid ---
valid_ok = by_id["valid"]["ok"]
invalid_ok = by_id["invalid"]["ok"]
print(f"coverageisvalid: valid_cov={valid_ok}, invalid_cov={invalid_ok}")
assert valid_ok is True, f"Expected valid coverage ok=True, got {valid_ok}"
assert invalid_ok is False, f"Expected invalid coverage ok=False, got {invalid_ok}"

stage1_pass = (valid_ok is True) and (invalid_ok is False)
results["stages"]["coverageisvalid"] = {
    "pass": stage1_pass,
    "valid_cov_result": bool(valid_ok),
    "invalid_cov_result": bool(invalid_ok),
}
print("PASS: coverageisvalid")

# --- Stage 2: coverageinvalidedges ---
valid_bad = by_id["valid"]["bad_edges"]
invalid_bad = by_id["invalid"]["bad_edges"]

# PySpark collect() returns bytearray for BinaryType; bytes() needed for shapely.
valid_edges_geom = from_wkb(bytes(valid_bad))
invalid_edges_geom = from_wkb(bytes(invalid_bad))

valid_edges_empty = valid_edges_geom.is_empty
invalid_edges_nonempty = not invalid_edges_geom.is_empty

print(f"coverageinvalidedges: valid_cov_empty={valid_edges_empty}, invalid_cov_nonempty={invalid_edges_nonempty}")
assert valid_edges_empty, f"Expected valid coverage to have empty invalid edges, got {valid_edges_geom}"
assert invalid_edges_nonempty, f"Expected invalid coverage to have non-empty invalid edges, got {invalid_edges_geom}"

stage2_pass = bool(valid_edges_empty) and bool(invalid_edges_nonempty)
results["stages"]["coverageinvalidedges"] = {
    "pass": stage2_pass,
    "valid_cov_edges_empty": bool(valid_edges_empty),
    "invalid_cov_has_edges": bool(invalid_edges_nonempty),
}
print("PASS: coverageinvalidedges")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 3: vx.coverage_simplify
# Two adjacent polygons sharing an edge.  Each has an extra collinear vertex
# at (1, 0.5) on the shared edge x=1 between y=0 and y=1.  After simplification
# with tolerance=0.1, that collinear point is removed from both polygons.
# Assert: N→N rows, all input columns preserved, total vertex count reduced,
# shared edge still intact (polygons touch; no overlap introduced).
# ---------------------------------------------------------------------------
left = Polygon([(0, 0), (1, 0), (1, 0.5), (1, 1), (0, 1)])
right = Polygon([(1, 0), (2, 0), (2, 1), (1, 1), (1, 0.5)])

# Exterior coords include closing coord; subtract 1 for distinct-vertex count.
n_input_left = len(left.exterior.coords) - 1    # 5
n_input_right = len(right.exterior.coords) - 1  # 5
n_input_total = n_input_left + n_input_right     # 10

rows_simplify = [
    ("s", "left", to_wkb(left)),
    ("s", "right", to_wkb(right)),
]
df_simp = spark.createDataFrame(rows_simplify, "cov_id string, name string, geom binary")

TOLERANCE = 0.1
out_rows = vx.coverage_simplify(df_simp, "cov_id", "geom", TOLERANCE).collect()

n_out = len(out_rows)
n_in = len(rows_simplify)
assert n_out == n_in, f"Expected N={n_in} rows out (N→N), got {n_out}"

# All input columns must be preserved; new out_col added.
orig_cols = {"cov_id", "name", "geom"}
out_cols = set(out_rows[0].asDict().keys())
columns_preserved = orig_cols <= out_cols
for col in orig_cols:
    assert col in out_cols, f"Input column '{col}' missing from output"
assert "geom_simplified" in out_cols, "Output column 'geom_simplified' missing"

out_by_name = {row["name"]: row for row in out_rows}
g_left_out = from_wkb(bytes(out_by_name["left"]["geom_simplified"]))
g_right_out = from_wkb(bytes(out_by_name["right"]["geom_simplified"]))
n_out_left = len(g_left_out.exterior.coords) - 1
n_out_right = len(g_right_out.exterior.coords) - 1
n_out_total = n_out_left + n_out_right

print(f"coverage_simplify (tol={TOLERANCE}):")
print(f"  input  total vertices={n_input_total} (left={n_input_left}, right={n_input_right})")
print(f"  output total vertices={n_out_total}  (left={n_out_left}, right={n_out_right})")
print(f"  names in output: {sorted(out_by_name.keys())}")

vertices_reduced = n_out_total < n_input_total
assert vertices_reduced, f"Expected vertex count to decrease, {n_input_total} -> {n_out_total}"

# Shared-edge topology: sort by x-min so left<right deterministically; they must
# touch (share a boundary) and must not overlap.
g0, g1 = sorted((g_left_out, g_right_out), key=lambda g: g.bounds[0])
shared_edge_intact = bool(g0.touches(g1))
no_overlap = g0.intersection(g1).area == 0.0
print(f"  shared_edge_intact={shared_edge_intact}, no_overlap={no_overlap}")
assert shared_edge_intact and no_overlap, (
    f"coverage_simplify broke shared-edge topology: touches={shared_edge_intact}, no_overlap={no_overlap}"
)

stage3_pass = vertices_reduced and columns_preserved and shared_edge_intact and no_overlap
results["stages"]["coverage_simplify"] = {
    "pass": stage3_pass,
    "n_rows_in": n_in,
    "n_rows_out": n_out,
    "n_input_total_vertices": n_input_total,
    "n_output_total_vertices": n_out_total,
    "columns_preserved": columns_preserved,
    "shared_edge_intact": shared_edge_intact,
    "no_overlap": no_overlap,
    "tolerance": TOLERANCE,
}
print(f"PASS: coverage_simplify — N→N ({n_in}->{n_out}), vertices {n_input_total}->{n_out_total}, topology intact")

In [ ]:
# ---------------------------------------------------------------------------
# Final: assemble results, assert overall pass, exit with structured output.
# ---------------------------------------------------------------------------
all_pass = all(
    results["stages"].get(fn, {}).get("pass", False)
    for fn in ["coverageisvalid", "coverageinvalidedges", "coverage_simplify"]
)
results["all_pass"] = all_pass

print(json.dumps(results, indent=2))

if all_pass:
    print("\nPASS: all coverage-validity validation checks green.")
    for fn, stage in results["stages"].items():
        print(f"  - {fn}: pass={stage['pass']}")
else:
    print("\nFAILED: see results dict above for details.")

assert all_pass, "Coverage-validity validation FAILED — see results dict for details."
dbutils.notebook.exit(json.dumps(results))